# LOFAR single station imaging - Video making

In [1]:
import numpy as np
import os
import re
import getpass
import datetime
import matplotlib.pyplot as plt
import pandas as pd
import glob
import cv2
import time

In [2]:
from lofarantpos.db import LofarAntennaDatabase
from lofarimaging import read_acm_cube, get_station_type, make_xst_plots

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
db = LofarAntennaDatabase()

In [5]:
username = getpass.getuser()
data_dir = f"/home/{username}/Documents/LV614_image_data/"
caltable_dir = f"/home/{username}/Documents/LV614_image_data/CalTables/"
caltable_dir = "CalTables/"

output_dir = "results/movie"
temp_dir = output_dir + "/temp"
fps = 10

# Observation that works
obs_dir = data_dir + "sess_sid20230111T071252_LV614/scan_59955.30061/LV614_20230111_071256_spw3_sb51_461_int1_dur24300_dir0,0,SUN_xst/"

# Observation info
station_name = "LV614"
integration_time_s = 2
rcu_mode = 3
height = 1.5
station_type = get_station_type(station_name)


In [6]:
def get_subbands(file_path):
    with open(file_path, 'r') as file:
        content = file.read()
    match = re.search(r'--xcsubband=(\d+)', content)
    return int(match.group(1)) if match else None

def get_obstime(file_path):
    obsdatestr, obstimestr, *_ = os.path.basename(file_path).rstrip(".dat").split("_")
    return datetime.datetime.strptime(obsdatestr + ":" + obstimestr, '%Y%m%d:%H%M%S')

def analyze_files(data_files):
    dat_files = sorted(glob.glob(os.path.join(data_files, '*.dat')))
    h_files = sorted(glob.glob(os.path.join(data_files, '*.h')))

    assert len(dat_files) == len(h_files), "Mismatch in number of .dat and .h files"

    data_list = []

    for dat_file, h_file in zip(dat_files, h_files):
        timestamp = get_obstime(dat_file)
        timestamp2 = get_obstime(h_file)
        if timestamp != timestamp2:
            print(f"Warning: timestamps do not match for {dat_file} and {h_file}")
        subband = get_subbands(h_file)

        data_list.append({
            "timestamp": timestamp,
            "subband": subband,
            "dat_file": dat_file,
            "h_file": h_file
        })

    df = pd.DataFrame(data_list)
    df = df.sort_values(by=["timestamp", "subband"]).reset_index(drop=True)

    average_measures_per_subband = round(df["subband"].value_counts().mean(), 2)
    measurement_duration = round((df["timestamp"].max() - df["timestamp"].min()).total_seconds() / len(df), 2) if len(df) > 1 else None

    summary = {
        "number_of_files": len(df),
        "subbands_available": {
            "first_subband": df["subband"].min(),
            "last_subband": df["subband"].max(),
            "total_subbands": len(df["subband"].dropna().unique())
        },
        "start_time": df["timestamp"].min(),
        "end_time": df["timestamp"].max(),
        "average_measures_per_subband": average_measures_per_subband,
        "measurement_duration": measurement_duration
    }

    return df, summary

def print_summary(summary):
    print("Summary of Analyzed Files:")
    print(f"Number of files: {summary['number_of_files']}")
    print(f"First and last subband: {summary['subbands_available']['first_subband']} - {summary['subbands_available']['last_subband']}")
    print(f"Total subbands: {summary['subbands_available']['total_subbands']}")
    print(f"Start time: {summary['start_time']}")
    print(f"End time: {summary['end_time']}")
    print(f"Average measurements per subband: {summary['average_measures_per_subband']}")
    print(f"Average measurement duration: {summary['measurement_duration']} seconds")


def generate_movie(sources, output_path, fps=10):
    if not sources:
        print("Image list is empty.")
        return

    # Use the first image to get the size of the frame
    image = cv2.imread(sources[0])
    if image is None:
        print(f"Error reading image: {sources[0]}. Aborting.")
        return

    height, width, _ = image.shape

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Iterate through the images and write them to the movie
    for path in sources:
        image = cv2.imread(path)
        if image is None:
            print(f"Error reading image: {path}. It will be skipped.")
            continue
        out.write(image)

    out.release()
    cv2.destroyAllWindows()


def generate_movie_from_list(list_path, output_path, fps=10):
    with open(list_path, "r") as file:
        sources = file.read().splitlines()
    generate_movie(sources, output_path, fps=fps)


In [7]:
df, summary = analyze_files(obs_dir)
print_summary(summary)
df

Summary of Analyzed Files:
Number of files: 12150
First and last subband: 51 - 461
Total subbands: 411
Start time: 2023-01-11 07:12:56
End time: 2023-01-11 13:57:54
Average measurements per subband: 29.56
Average measurement duration: 2.0 seconds


,timestamp,subband,dat_file,h_file
0,2023-01-11 07:12:56,51,/home/jan/Documents/LV614_image_data/sess_sid2...,/home/jan/Documents/LV614_image_data/sess_sid2...
1,2023-01-11 07:12:58,52,/home/jan/Documents/LV614_image_data/sess_sid2...,/home/jan/Documents/LV614_image_data/sess_sid2...
2,2023-01-11 07:13:00,53,/home/jan/Documents/LV614_image_data/sess_sid2...,/home/jan/Documents/LV614_image_data/sess_sid2...
3,2023-01-11 07:13:02,54,/home/jan/Documents/LV614_image_data/sess_sid2...,/home/jan/Documents/LV614_image_data/sess_sid2...
4,2023-01-11 07:13:04,55,/home/jan/Documents/LV614_image_data/sess_sid2...,/home/jan/Documents/LV614_image_data/sess_sid2...
...,...,...,...,...
12145,2023-01-11 13:57:46,277,/home/jan/Documents/LV614_image_data/sess_sid2...,/home/jan/Documents/LV614_image_data/sess_sid2...
12146,2023-01-11 13:57:48,278,/home/jan/Documents/LV614_image_data/sess_sid2...,/home/jan/Documents/LV614_image_data/sess_sid2...
12147,2023-01-11 13:57:50,279,/home/jan/Documents/LV614_image_data/sess_sid2...,/home/jan/Documents/LV614_image_data/sess_sid2...
12148,2023-01-11 13:57:52,280,/home/jan/Documents/LV614_image_data/sess_sid2...,/home/jan/Documents/LV614_image_data/sess_sid2...


In [8]:
# Temporal evolution of the selected subbands

height = 1.5
subbands = [200, 280, 360]
#subbands = [255]
sky_movie = []
nf_movie = []

for subband in subbands:
    subband_data = df[df['subband'] == subband]#.head(3)

    for index, row in subband_data.iterrows():
        xst_filename = row['dat_file']
        obstime = row['timestamp']
        time.sleep(0.1)

        try:
            visibilities = read_acm_cube(xst_filename, station_type)[0]
            #sky_fig, ground_fig, leaflet_map = make_xst_plots(visibilities, station_name, obstime, subband, rcu_mode, map_zoom=18, outputpath=temp_dir, mark_max_power=True, height=height)
            sky_image_path, nf_image_path, _ = make_xst_plots(visibilities, station_name, obstime, subband, rcu_mode, map_zoom=18, outputpath=temp_dir, mark_max_power=True, height=height, return_only_paths=True)
            sky_movie.append(sky_image_path)
            nf_movie.append(nf_image_path)
        except Exception as e:
            print(f"Error generating image for {xst_filename}: {e}")

# Export the image lists for movie generation
with open(f"{temp_dir}/sky_image_list.txt", "w") as sky_file:
    sky_file.write("\n".join(sky_movie))

with open(f"{temp_dir}/nf_image_list.txt", "w") as nf_file:
    nf_file.write("\n".join(nf_movie))

print("Image generation complete.")

generate_movie_from_list(f"{temp_dir}/sky_image_list.txt", f"{output_dir}/sky_movie_time_sweep.mp4", fps=fps)
generate_movie_from_list(f"{temp_dir}/nf_image_list.txt", f"{output_dir}/nf_movie_time_sweep.mp4", fps=fps)

print("Movie generation complete.")

Maximum of 60.48 dB at 48m east, -72m north of station center (lat/long 57.55623, 21.85572)
Maximum of 60.49 dB at 68m east, -112m north of station center (lat/long 57.55587, 21.85605)
Maximum of 60.52 dB at 74m east, -150m north of station center (lat/long 57.55553, 21.85616)
Maximum of 60.63 dB at 108m east, -150m north of station center (lat/long 57.55553, 21.85672)
Maximum of 60.62 dB at 46m east, -150m north of station center (lat/long 57.55553, 21.85569)
Maximum of 60.88 dB at 90m east, -150m north of station center (lat/long 57.55553, 21.85642)
Maximum of 60.76 dB at 24m east, -150m north of station center (lat/long 57.55553, 21.85532)
Maximum of 60.63 dB at 6m east, -150m north of station center (lat/long 57.55553, 21.85502)
Maximum of 60.84 dB at 0m east, -150m north of station center (lat/long 57.55553, 21.85492)
Maximum of 60.73 dB at -2m east, -150m north of station center (lat/long 57.55553, 21.85489)
Maximum of 61.67 dB at 38m east, -150m north of station center (lat/long

In [9]:
# Subband sweep at a given time

height = 1.5

subband_min = 180
subband_max = 460
subband_step = 5
subbands = df["subband"].unique()
subbands = [s for s in sorted(subbands) if subband_min <= s <= subband_max][::subband_step]

min_time = df["timestamp"].min()
times = [
    #min_time + datetime.timedelta(minutes=5),
    min_time + datetime.timedelta(minutes=50),
    min_time + datetime.timedelta(minutes=200)
]
sky_movie = []
nf_movie = []

for t in times:

    for subband in subbands:
        subband_data = df[df['subband'] == subband]

        subband_data = subband_data[subband_data['timestamp'] >= t]
        subband_data = subband_data.sort_values('timestamp')
        closest_row = subband_data.iloc[0] if not subband_data.empty else None

        if closest_row is None:
            print(f"No data found for subband {subband} at {t}")
            continue

        xst_filename = closest_row['dat_file']
        obstime = closest_row['timestamp']

        try:
            visibilities = read_acm_cube(xst_filename, station_type)[0]
            #sky_fig, ground_fig, leaflet_map = make_xst_plots(visibilities, station_name, obstime, subband, rcu_mode, map_zoom=18, outputpath=temp_dir, mark_max_power=True, height=height)
            sky_image_path, nf_image_path, _ = make_xst_plots(visibilities, station_name, obstime, subband, rcu_mode, map_zoom=18, outputpath=temp_dir, mark_max_power=True, height=height, return_only_paths=True)
            sky_movie.append(sky_image_path)
            nf_movie.append(nf_image_path)
        except Exception as e:
            print(f"Error generating image for {xst_filename}: {e}")

# Export the image lists for movie generation
with open(f"{temp_dir}/sky_image_list.txt", "w") as sky_file:
    sky_file.write("\n".join(sky_movie))

with open(f"{temp_dir}/nf_image_list.txt", "w") as nf_file:
    nf_file.write("\n".join(nf_movie))

print("Image generation complete.")

generate_movie_from_list(f"{temp_dir}/sky_image_list.txt", f"{output_dir}/sky_movie_subband_sweep.mp4", fps=fps)
generate_movie_from_list(f"{temp_dir}/nf_image_list.txt", f"{output_dir}/nf_movie_subband_sweep.mp4", fps=fps)

print("Movie generation complete.")

Maximum of 59.77 dB at 54m east, -150m north of station center (lat/long 57.55553, 21.85582)
Maximum of 59.93 dB at 54m east, -150m north of station center (lat/long 57.55553, 21.85582)
Maximum of 60.07 dB at 48m east, -150m north of station center (lat/long 57.55553, 21.85572)
Maximum of 60.33 dB at 44m east, -150m north of station center (lat/long 57.55553, 21.85565)
Maximum of 60.62 dB at 46m east, -150m north of station center (lat/long 57.55553, 21.85569)
Maximum of 60.89 dB at 42m east, -150m north of station center (lat/long 57.55553, 21.85562)
Maximum of 61.16 dB at 42m east, -150m north of station center (lat/long 57.55553, 21.85562)
Maximum of 61.47 dB at 44m east, -150m north of station center (lat/long 57.55553, 21.85565)
Maximum of 61.97 dB at 148m east, -72m north of station center (lat/long 57.55623, 21.85739)
Maximum of 62.32 dB at 40m east, -150m north of station center (lat/long 57.55553, 21.85559)
Maximum of 62.68 dB at 40m east, -150m north of station center (lat/lo

In [10]:
# Height sweep at a given time and subband